# Pour le quatrième et dernier pipeline, je voulais vérifier si la technique d’embedding influence les performances du modèle. J’ai donc utilisé Word2Vec pour générer les embeddings.

In [1]:
import pandas as pd
import regex as re
import contractions
import spacy

In [3]:
raw = pd.read_csv("../IMDB Dataset.csv")
raw = raw.drop_duplicates(subset='review', keep='first')

In [4]:
raw

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


# Prétraitement


In [5]:
raw[raw['review'].str.contains(r'\b[\w\.-]+@[\w\.-]+\.\w+\b')]

,review,sentiment
1281,I like many others saw this as a child and I l...,positive
3568,I have noticed that people have asked if anyon...,positive
5068,Brilliant adaptation of the largely interior m...,positive
8176,I could never remember the name of this show. ...,positive
8474,I'm like the rest of the fans who love this co...,positive
9510,I cant believe how many excellent actors can b...,positive
9978,I really can't say too much more about the plo...,positive
12166,Robert Jordan is a television star. Robert Jor...,positive
13471,This movie was everything but boring. It deals...,positive
13741,I LOVE this movie. and Disney channel is ridic...,positive


In [6]:
cleaning = raw.copy()

In [7]:
cleaning["review"] = cleaning["review"].apply(lambda x: re.sub(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', ' ', x))
cleaning["review"] = cleaning["review"].apply(lambda x: re.sub(r"<.*?>", "", x))
cleaning["review"] = cleaning["review"].apply(lambda x: x.lower())
cleaning["review"] = cleaning["review"].apply(lambda x: contractions.fix(x))
nlp = spacy.load("en_core_web_sm")
cleaning["review"] = cleaning["review"].apply(lambda x: " ".join(token.lemma_ for token in nlp(x)))


In [8]:
cleaning["label"] = cleaning["sentiment"] == "positive"

In [9]:
cleaned = cleaning[["review", "label"]].copy()

# Tokenization & vectorization

In [10]:
import numpy as np
from gensim.models import Word2Vec
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

In [11]:
# Tokenize the reviews (split into words)
cleaned["tokens"] = cleaned["review"].apply(lambda x: x.split())

# Train Word2Vec model
word2vec_model = Word2Vec(
    sentences=cleaned["tokens"].tolist(),
    vector_size=100,  # Embedding dimension
    window=5,         # Context window
    min_count=2,      # Ignore words with frequency less than this
    workers=4,
    epochs=10
)

In [12]:
# Create word-to-index mapping
vocab = word2vec_model.wv.key_to_index
vocab_size = len(vocab) + 1  # +1 for padding token

In [22]:
vocab_size

60089

In [13]:
# Create embedding matrix
embedding_dim = 100
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, idx in vocab.items():
    embedding_matrix[idx] = word2vec_model.wv[word]

In [23]:
embedding_matrix

array([[-4.67822433e-01, -7.88789093e-01,  4.23912883e-01, ...,
        -1.80737424e+00, -1.20281744e+00,  6.93491638e-01],
       [-2.18363017e-01, -1.87654400e+00,  8.62342775e-01, ...,
         1.69539452e+00, -6.60667717e-01,  4.29124743e-01],
       [-5.78907430e-01, -5.40357232e-01, -5.50038964e-02, ...,
         2.13998032e+00, -2.41483152e-01, -5.29237054e-02],
       ...,
       [ 5.06140925e-02,  2.10678447e-02,  1.01396246e-02, ...,
        -1.34710565e-01,  6.46122098e-02, -3.93607654e-02],
       [ 1.21557037e-03,  1.21355079e-01, -1.50293410e-02, ...,
         4.67071459e-02, -3.75534333e-02, -5.98629341e-02],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00]],
      shape=(60089, 100))

In [14]:
# Convert tokens to sequences of indices
def tokens_to_sequences(tokens):
    return [vocab.get(word, 0) for word in tokens]

cleaned["sequences"] = cleaned["tokens"].apply(tokens_to_sequences)

In [24]:
cleaned

,review,label,tokens,sequences
0,one of the other reviewer have mention that af...,True,"[one, of, the, other, reviewer, have, mention,...","[32, 6, 0, 82, 1122, 15, 424, 12, 114, 67, 48,..."
1,a wonderful little production . the filming te...,True,"[a, wonderful, little, production, ., the, fil...","[5, 408, 128, 341, 3, 0, 2073, 1652, 1, 65, 0,..."
2,I think this be a wonderful way to spend time ...,True,"[I, think, this, be, a, wonderful, way, to, sp...","[8, 78, 11, 1, 5, 408, 103, 7, 452, 59, 26, 5,..."
3,basically there be a family where a little boy...,False,"[basically, there, be, a, family, where, a, li...","[691, 46, 1, 5, 224, 126, 5, 128, 292, 27, 301..."
4,"petter mattei 's "" love in the time of money ""...",True,"[petter, mattei, 's, "", love, in, the, time, o...","[0, 9219, 29, 14, 102, 10, 0, 59, 6, 294, 14, ..."
...,...,...,...,...
49995,I think this movie do a down right good job . ...,True,"[I, think, this, movie, do, a, down, right, go...","[8, 78, 11, 17, 22, 5, 186, 202, 42, 274, 3, 9..."
49996,"bad plot , bad dialogue , bad acting , idiotic...",False,"[bad, plot, ,, bad, dialogue, ,, bad, acting, ...","[70, 118, 2, 70, 403, 2, 70, 163, 2, 2718, 141..."
49997,I be a catholic teach in parochial elementary ...,False,"[I, be, a, catholic, teach, in, parochial, ele...","[8, 1, 5, 2803, 1330, 10, 31583, 6958, 366, 37..."
49998,I be go to have to disagree with the previous ...,False,"[I, be, go, to, have, to, disagree, with, the,...","[8, 1, 64, 7, 15, 7, 2543, 20, 0, 890, 416, 4,..."


In [15]:
# Pad sequences to have the same length
max_length = 200  # You can adjust this based on your data
X = pad_sequences(cleaned["sequences"].tolist(), maxlen=max_length, padding='post')
y = cleaned["label"].astype(int).values

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [16]:
 # Build LSTM model
model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        trainable=True  # Word2Vec embeddings learns in training
    ),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(32)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

2025-11-09 22:02:29.197392: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [17]:
# Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [18]:
# Print model summary
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     6,008,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,008,900 (22.92 MB)

 Trainable params: 6,008,900 (22.92 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
# Train the model
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/20
496/496 ━━━━━━━━━━━━━━━━━━━━ 104s 200ms/step - accuracy: 0.7765 - loss: 0.4821 - val_accuracy: 0.8453 - val_loss: 0.3703
Epoch 2/20
496/496 ━━━━━━━━━━━━━━━━━━━━ 132s 266ms/step - accuracy: 0.8720 - loss: 0.3162 - val_accuracy: 0.8765 - val_loss: 0.3040
Epoch 3/20
496/496 ━━━━━━━━━━━━━━━━━━━━ 102s 205ms/step - accuracy: 0.9079 - loss: 0.2402 - val_accuracy: 0.8792 - val_loss: 0.3015
Epoch 4/20
496/496 ━━━━━━━━━━━━━━━━━━━━ 105s 211ms/step - accuracy: 0.9402 - loss: 0.1685 - val_accuracy: 0.8889 - val_loss: 0.3378
Epoch 5/20
496/496 ━━━━━━━━━━━━━━━━━━━━ 108s 218ms/step - accuracy: 0.9630 - loss: 0.1110 - val_accuracy: 0.8795 - val_loss: 0.3898
Epoch 6/20
496/496 ━━━━━━━━━━━━━━━━━━━━ 107s 215ms/step - accuracy: 0.8801 - loss: nan - val_accuracy: 0.4987 - val_loss: nan


In [20]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=1)
print(f"\nTest Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

# Make predictions
predictions = model.predict(X_test)
predicted_labels = (predictions > 0.5).astype(int)

310/310 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - accuracy: 0.8887 - loss: 0.2831

Test Accuracy: 0.8887
Test Loss: 0.2831
310/310 ━━━━━━━━━━━━━━━━━━━━ 9s 28ms/step


In [21]:
# Optional: Print classification report
from sklearn.metrics import classification_report, confusion_matrix

print("\nClassification Report:")
print(classification_report(y_test, predicted_labels, target_names=['Negative', 'Positive']))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predicted_labels))


Classification Report:
              precision    recall  f1-score   support

    Negative       0.91      0.86      0.89      4940
    Positive       0.87      0.91      0.89      4977

    accuracy                           0.89      9917
   macro avg       0.89      0.89      0.89      9917
weighted avg       0.89      0.89      0.89      9917


Confusion Matrix:
[[4266  674]
 [ 430 4547]]


In [25]:
import pickle
from tensorflow.keras.models import save_model

# 1. Save the trained LSTM model
model.save('sentiment_lstm_word2vec_model.h5')

# 2. Save the Word2Vec model
word2vec_model.save('word2vec_model.bin')

# 3. Save the vocabulary mapping and max_length
model_config = {
    'vocab': vocab,
    'vocab_size': vocab_size,
    'max_length': max_length,
    'embedding_dim': embedding_dim
}

with open('lstm_word2vec_model_config.pkl', 'wb') as f:
    pickle.dump(model_config, f)

print("All models and configurations saved successfully!")

All models and configurations saved successfully!
